# 62. 独립검증 후보의 데이터 사용 이력 감사
원본 labeled_data.csv에서 기존591개RG3 RH키와겹치지않는37개 기록을 메타데이터만으로 찾는다. 정답·센서값은열지않는다. 기존보고서의해당시각사용흔적을검색한다. 저장된흔적부재는과거의모든비저장조회가없었다는절대증명은아니다. 기존오리지널센서이력에서테스트센서가관측됐을가능성도구분한다.


In [ ]:
from pathlib import Path
import pandas as pd,json,hashlib,zipfile
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
cols=['EQUIP_CD','PART_NAME','TimeStamp']
raw=pd.read_csv(root/'data/original/labeled_data.csv',usecols=cols)
raw.TimeStamp=pd.to_datetime(raw.TimeStamp)
old=pd.read_csv(root/'data/audit/rg3_rh_labeled_with_timestamp.csv',usecols=cols)
old.TimeStamp=pd.to_datetime(old.TimeStamp)
rg=raw[raw.PART_NAME.str.contains('RG3',na=False)&raw.PART_NAME.str.contains('RH',na=False)]
new=rg.merge(old.assign(previously_used_key=True),on=cols,how='left')
new=new[new.previously_used_key.isna()].drop(columns='previously_used_key').sort_values(cols).reset_index(drop=True)
assert len(new)==37 and not new.duplicated(cols).any()
assert new.TimeStamp.min()>old.TimeStamp.max()
strings=set(new.TimeStamp.dt.strftime('%Y-%m-%d %H:%M:%S'))
matches=[]
for path in tables.glob('*.csv'):
 if path.name.startswith('rg3_independent_'):continue
 text=path.read_text(encoding='utf-8-sig',errors='replace')
 if any(x in text for x in strings):matches.append(str(path.relative_to(root)))
new.to_csv(tables/'rg3_independent_candidate_keys.csv',index=False,encoding='utf-8-sig')
inventory=[]
for path in (root/'data').rglob('*.csv'):
 headers=pd.read_csv(path,nrows=0).columns.tolist()
 inventory.append(dict(path=str(path.relative_to(root)),bytes=path.stat().st_size,has_shot_quality_label='PassOrFail' in headers,has_reason='Reason' in headers,has_timestamp='TimeStamp' in headers))
pd.DataFrame(inventory).to_csv(tables/'rg3_independent_data_inventory.csv',index=False,encoding='utf-8-sig')
print('EXTRA RG3 RH metadata only:',len(new),'from',new.TimeStamp.min(),'to',new.TimeStamp.max())
print('Prior report timestamp matches:',matches)
print('No extra candidate labels or sensor values were opened by this audit.')
print('Evidence is absence of stored use traces, not proof that nobody has ever viewed this source interactively.')
